# Wearwell 백엔드 — Google Colab L4

이 노트북은 `dongwgo/wearwell`을 clone하고, **FASHN VTON v1.5**와 SDXL Turbo를 설치한 뒤 FastAPI 서버와 Cloudflare Quick Tunnel을 실행합니다. 마지막 셀은 공개 백엔드 URL을 `config.js`에 기록하고 출력합니다.

> Colab GPU 메뉴에는 일반적으로 **G4가 아니라 L4**가 표시됩니다. 유료 런타임에서 `런타임 → 런타임 유형 변경 → L4 GPU`를 선택하세요. A100도 지원하며, T4에서는 BF16 대신 FP32로 동작해 느리고 메모리가 부족할 수 있습니다.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError(f"Python 3.10+가 필요합니다. 현재: {sys.version}")

print("Python:", sys.version.split()[0])
subprocess.run(["nvidia-smi"], check=True)

try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("GPU 런타임이 아닙니다. Colab 런타임 유형을 L4 GPU로 바꾸세요.")
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if not any(name in gpu_name for name in ("L4", "A100", "H100")):
        print("경고: L4/A100/H100이 아닙니다. 실행은 가능하지만 속도나 VRAM이 부족할 수 있습니다.")
except ImportError as exc:
    raise RuntimeError("Colab GPU 런타임의 PyTorch를 찾지 못했습니다.") from exc


In [ ]:
import shutil

REPO_URL = "https://github.com/dongwgo/wearwell.git"
REPO_DIR = Path("/content/wearwell")
FASHN_DIR = Path("/content/fashn-vton-1.5")
FASHN_COMMIT = "7c0f10af3f91ad4048fe9729c470a13ef905d25a"
WEIGHTS_DIR = Path("/content/models/fashn-vton-1.5")
HF_HOME = Path("/content/hf-cache")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)

def run(*args, cwd=None):
    print("+", " ".join(map(str, args)))
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)

if (REPO_DIR / ".git").exists():
    run("git", "checkout", "--", "config.js", cwd=REPO_DIR)
    run("git", "pull", "--ff-only", cwd=REPO_DIR)
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run("git", "clone", REPO_URL, REPO_DIR)

if not (FASHN_DIR / ".git").exists():
    run("git", "clone", "https://github.com/fashn-AI/fashn-vton-1.5.git", FASHN_DIR)
run("git", "fetch", "origin", FASHN_COMMIT, cwd=FASHN_DIR)
run("git", "checkout", "--detach", FASHN_COMMIT, cwd=FASHN_DIR)

run(sys.executable, "-m", "pip", "install", "-q", "-r", REPO_DIR / "backend/requirements.txt")
run(sys.executable, "-m", "pip", "install", "-q", "-e", FASHN_DIR)

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
required_weights = [
    WEIGHTS_DIR / "model.safetensors",
    WEIGHTS_DIR / "dwpose/yolox_l.onnx",
    WEIGHTS_DIR / "dwpose/dw-ll_ucoco_384.onnx",
]
if not all(path.is_file() and path.stat().st_size > 0 for path in required_weights):
    run(sys.executable, FASHN_DIR / "scripts/download_weights.py", "--weights-dir", WEIGHTS_DIR)
else:
    print("FASHN weights already present:", WEIGHTS_DIR)


In [ ]:
import json
import secrets
import time
import urllib.request

BACKEND_PORT = 8787
API_TOKEN = secrets.token_urlsafe(32)
SERVER_LOG = Path("/content/wearwell-uvicorn.log")

for name in ("wearwell_server", "wearwell_tunnel"):
    process = globals().get(name)
    if process and process.poll() is None:
        process.terminate()
        process.wait(timeout=10)

env = os.environ.copy()
env.update({
    "FASHN_WEIGHTS_DIR": str(WEIGHTS_DIR),
    "HF_HOME": str(HF_HOME),
    "AVATAR_MODEL": "stabilityai/sdxl-turbo",
    "FASHN_STEPS": "30",
    "ONEULOUT_GPU": "1",
    "WEARWELL_API_TOKEN": API_TOKEN,
    "PYTHONUNBUFFERED": "1",
})

server_log_handle = SERVER_LOG.open("w")
wearwell_server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", str(BACKEND_PORT)],
    cwd=REPO_DIR / "backend",
    env=env,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
)

health_url = f"http://127.0.0.1:{BACKEND_PORT}/api/health"
for _ in range(60):
    if wearwell_server.poll() is not None:
        raise RuntimeError(SERVER_LOG.read_text(errors="replace"))
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            health = json.load(response)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("백엔드 시작 시간 초과\n" + SERVER_LOG.read_text(errors="replace"))

warmup_request = urllib.request.Request(
    f"http://127.0.0.1:{BACKEND_PORT}/api/warmup",
    method="POST",
    headers={"Authorization": f"Bearer {API_TOKEN}"},
)
print("모델 다운로드·로드 검증 중 (첫 실행은 수 분 걸립니다)…")
with urllib.request.urlopen(warmup_request, timeout=1800) as response:
    print(json.dumps(json.load(response), ensure_ascii=False, indent=2))
with urllib.request.urlopen(health_url, timeout=5) as response:
    health = json.load(response)
if not health.get("warmupVerified"):
    raise RuntimeError("Model warmup verification failed")
print(json.dumps(health, ensure_ascii=False, indent=2))


In [ ]:
import hashlib
import re

CLOUDFLARED = Path("/content/cloudflared")
CLOUDFLARED_VERSION = "2026.8.2"
CLOUDFLARED_SHA256 = "fcfb02b575a52ca1af2e3267af4e1517bcdeb30ac48c834c69abaed3c0576ad2"
TUNNEL_LOG = Path("/content/cloudflared.log")
if CLOUDFLARED.exists() and hashlib.sha256(CLOUDFLARED.read_bytes()).hexdigest() != CLOUDFLARED_SHA256:
    CLOUDFLARED.unlink()
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
    if hashlib.sha256(CLOUDFLARED.read_bytes()).hexdigest() != CLOUDFLARED_SHA256:
        CLOUDFLARED.unlink(missing_ok=True)
        raise RuntimeError("cloudflared checksum mismatch")
CLOUDFLARED.chmod(0o755)

tunnel_log_handle = TUNNEL_LOG.open("w")
wearwell_tunnel = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{BACKEND_PORT}", "--no-autoupdate"],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)

backend_url = None
for _ in range(90):
    if wearwell_tunnel.poll() is not None:
        raise RuntimeError(TUNNEL_LOG.read_text(errors="replace"))
    text = TUNNEL_LOG.read_text(errors="replace") if TUNNEL_LOG.exists() else ""
    match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", text)
    if match:
        backend_url = match.group(0)
        break
    time.sleep(1)
if not backend_url:
    raise RuntimeError("Cloudflare Tunnel URL 생성 시간 초과\n" + TUNNEL_LOG.read_text(errors="replace"))

config_path = REPO_DIR / "config.js"
config = config_path.read_text(encoding="utf-8")
config = re.sub(r'API_BASE:\s*"[^"]*"', f'API_BASE: "{backend_url}"', config, count=1)
config_path.write_text(config, encoding="utf-8")

from IPython.display import HTML, display
app_url = f"{backend_url}/#token={API_TOKEN}"
print("BACKEND_URL=" + backend_url)
print("APP_URL=" + app_url)
print("config.js updated:", config_path)
display(HTML(f'<p><a href="{app_url}" target="_blank" style="font-size:20px;font-weight:700">인증된 Wearwell 열기</a></p>'))


## 사용 방법

- 마지막 셀의 **인증된 Wearwell 열기** 링크(`APP_URL`)를 사용하세요.
- 링크의 `#token=...` fragment는 Cloudflare 서버에 전송되지 않고 브라우저에서 API 인증에만 사용됩니다.
- URL은 Cloudflare Quick Tunnel의 임시 주소라서 Colab 런타임을 재시작하면 바뀝니다. 재시작할 때 마지막 두 셀을 다시 실행하세요.
- 인증 토큰은 실행할 때마다 새로 만들며 notebook이나 Git에 저장하지 않습니다.
